# 04 · Temporal Features & User Histories

**Goal.** Turn each post's timestamp into useful numbers, and group posts by user into **time-ordered
histories**.

**Why it matters.** When and how often someone posts carries signal, and a user's *sequence* of posts is
what later lets us measure language "drift" over time.

**What happens below.** We compute time-of-day / day-of-week features, and build per-user post timelines
for the users who have enough history.

**Output:** `data/stage1_temporal/` (time features + per-user histories).

## 0. Environment & dependencies

Core (`numpy`, `pandas`) always load. No new heavy dependency — MinMax is computed manually (transparent,
and it leaves the already-binary features untouched). Same defensive-flag pattern as Parts 1.1–1.3.

In [1]:
import os, re, json, math, logging, warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

def _try(name):
    try:
        __import__(name); return True
    except Exception:
        return False

HAS_PYARROW = _try("pyarrow") or _try("fastparquet")

SEED = 42
np.random.seed(SEED)
PART_VERSION = "1.0"

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("temporal")
log.info("deps -> parquet_engine=%s", HAS_PYARROW)

2026-08-07 21:26:36,963 | INFO | deps -> parquet_engine=True


## 1. Configuration

Platform roles, the six canonical temporal features (Saeed & Cha 2025 — the exact ablated set),
the D1 drift viability floor, the D2 chunk count, and the Couto full-matrix requirement. Nothing here
needs editing for a normal run.

In [2]:
def find_project_root():
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / "data" / "stage1_augmented").exists() or (d / ".env").exists():
            return d
    return here

PROJECT_ROOT = find_project_root()
AUG_DIR    = PROJECT_ROOT / "data" / "stage1_augmented"
TRAIN_IN   = AUG_DIR / "train_augmented.jsonl"
TEST_IN    = AUG_DIR / "test_frozen.jsonl"
OUTPUT_DIR = PROJECT_ROOT / "data" / "stage1_temporal"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# platform roles: YouTube = TRAIN; Reddit + Twitter = held-out TEST + drift source
TRAIN_PLATFORMS = {"youtube"}
DRIFT_PLATFORMS = {"reddit", "twitter"}

# the six canonical timestamp features (Saeed & Cha 2025), MinMax -> [0,1]
TEMPORAL_FEATURES = ["month", "day", "hour", "weekday", "is_working_hour", "is_weekend"]

# D1 — drift viability floor (keep users with >= this many posts for the scaffold)
DRIFT_MIN_POSTS = 3
# D2 — Couto chunk count (adaptive: min(MAX_CHUNKS, n_posts_user))
MAX_CHUNKS = 10
# the full (non-bounded) Couto delta-matrix wants >= this many posts per user
FULL_COUTO_MIN_POSTS = 10
# how many full-viable users we need before we stop calling it a "bounded case study"
FULL_COUTO_MIN_USERS = 5

TIER_NAMES = {0: "Neutral", 1: "Stress", 2: "Moderate", 3: "Crisis"}
N_LABELS = 4

log.info("PROJECT_ROOT=%s", PROJECT_ROOT)
log.info("TRAIN_IN=%s", TRAIN_IN)
log.info("TEST_IN=%s", TEST_IN)
log.info("D1 drift floor=%d posts | D2 chunks=min(%d,n) | full-Couto needs >=%d posts",
         DRIFT_MIN_POSTS, MAX_CHUNKS, FULL_COUTO_MIN_POSTS)

2026-08-07 21:26:36,974 | INFO | PROJECT_ROOT=hinglish-mental-health-detection


2026-08-07 21:26:36,975 | INFO | TRAIN_IN=hinglish-mental-health-detection\data\stage1_augmented\train_augmented.jsonl


2026-08-07 21:26:36,975 | INFO | TEST_IN=hinglish-mental-health-detection\data\stage1_augmented\test_frozen.jsonl


2026-08-07 21:26:36,977 | INFO | D1 drift floor=3 posts | D2 chunks=min(10,n) | full-Couto needs >=10 posts


## 2. Load the notebook 03 augmented train + frozen test

JSONL is the source of truth. The augmented train carries 167 real YouTube rows + 35 synthetic rows
(null timestamps); the frozen test carries the Reddit+Twitter held-out set (0 synthetic). The six
temporal features + `timestamp` + `is_synthetic` already exist on every row.

In [3]:
def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return pd.DataFrame(rows)

train_df = load_jsonl(TRAIN_IN)
test_df  = load_jsonl(TEST_IN)

for df in (train_df, test_df):
    if "is_synthetic" not in df.columns:
        df["is_synthetic"] = False
    df["is_synthetic"] = df["is_synthetic"].fillna(False).astype(bool)

log.info("train_augmented = %d rows (real %d + synthetic %d)",
         len(train_df), int((~train_df["is_synthetic"]).sum()), int(train_df["is_synthetic"].sum()))
log.info("test_frozen     = %d rows %s", len(test_df), dict(test_df["platform"].value_counts()))
print("train platforms:", dict(train_df["platform"].value_counts()))

2026-08-07 21:26:37,212 | INFO | train_augmented = 1971 rows (real 1846 + synthetic 125)


2026-08-07 21:26:37,219 | INFO | test_frozen     = 2768 rows {'twitter': np.int64(1466), 'reddit': np.int64(1302)}


train platforms: {'youtube': np.int64(1846), 'synthetic': np.int64(125)}


## 3. Tier-A — recompute the six features from `timestamp`, then MinMax **refit on TRAIN only**

We recompute the six features deterministically from each real `timestamp` (so the canonical matrix does
not depend on harvest-time scaling), then fit a MinMax scaler **on the YouTube TRAIN real rows only** and
apply it to the frozen test — standard no-leakage practice. We keep MinMax (not cyclic sin/cos) to
stay faithful to the cited Saeed & Cha recipe; the original harvest features remain untouched in
`train_augmented.jsonl` / `test_frozen.jsonl` for provenance. Synthetic / timestamp-less rows get
feature `0.0` placeholders that the **mask** (Step 4) tells every consumer to ignore.

In [4]:
def raw_temporal(ts):
    # returns the six RAW features from an ISO timestamp, or None for null/unparseable (synthetic)
    if ts is None or (isinstance(ts, float) and math.isnan(ts)) or str(ts).strip() in ("", "None", "nan"):
        return None
    try:
        dt = pd.to_datetime(ts, utc=True)
    except Exception:
        return None
    if pd.isna(dt):
        return None
    hour = int(dt.hour); wd = int(dt.weekday())
    return {
        "month": int(dt.month),
        "day": int(dt.day),
        "hour": hour,
        "weekday": wd,
        "is_working_hour": 1 if 9 <= hour < 18 else 0,
        "is_weekend": 1 if wd >= 5 else 0,
    }

def add_raw_features(df):
    recs = [raw_temporal(ts) for ts in df["timestamp"].tolist()]
    has_time = [r is not None for r in recs]
    for feat in TEMPORAL_FEATURES:
        df["raw_" + feat] = [(r[feat] if r is not None else np.nan) for r in recs]
    df["_has_time"] = has_time
    return df

train_df = add_raw_features(train_df)
test_df  = add_raw_features(test_df)

# fit MinMax on TRAIN real (has-time, non-synthetic) rows only
fit_mask = train_df["_has_time"] & (~train_df["is_synthetic"])
SCALER = {}
for feat in TEMPORAL_FEATURES:
    col = pd.to_numeric(train_df.loc[fit_mask, "raw_" + feat], errors="coerce").dropna()
    lo, hi = (float(col.min()), float(col.max())) if len(col) else (0.0, 1.0)
    SCALER[feat] = (lo, hi)

def apply_scaler(df):
    for feat in TEMPORAL_FEATURES:
        lo, hi = SCALER[feat]
        raw = pd.to_numeric(df["raw_" + feat], errors="coerce")
        if hi > lo:
            scaled = (raw - lo) / (hi - lo)
        else:
            scaled = raw * 0.0           # degenerate (e.g. all-same / binary collapsed) -> 0.0
        scaled = scaled.clip(0.0, 1.0)   # frozen test may exceed TRAIN range -> clamp (no leakage)
        df[feat + "_t"] = scaled.fillna(0.0)   # null/synthetic -> 0.0 placeholder (mask will ignore)
    return df

train_df = apply_scaler(train_df)
test_df  = apply_scaler(test_df)
log.info("MinMax fit on %d TRAIN real rows; scaler ranges=%s", int(fit_mask.sum()),
         {k: (round(v[0], 1), round(v[1], 1)) for k, v in SCALER.items()})

2026-08-07 21:26:40,102 | INFO | MinMax fit on 1846 TRAIN real rows; scaler ranges={'month': (1.0, 12.0), 'day': (1.0, 31.0), 'hour': (0.0, 23.0), 'weekday': (0.0, 6.0), 'is_working_hour': (0.0, 1.0), 'is_weekend': (0.0, 1.0)}


## 4. Tier-A — the `temporal_mask` (Invariant #1)

`temporal_mask = 1` iff the row has a real parsed timestamp **and** is not synthetic; else `0`. The
Objective 3 temporal stream and any Drift-Velocity computation must consume this mask and skip masked rows.
We assert the masked-out count equals the notebook 03 synthetic count (35) — the temporal twin of notebook 03's
leakage guard.

In [5]:
for df in (train_df, test_df):
    df["temporal_mask"] = (df["_has_time"] & (~df["is_synthetic"])).astype(int)

n_syn_train = int(train_df["is_synthetic"].sum())
n_masked_train = int((train_df["temporal_mask"] == 0).sum())

# every synthetic row must be masked out
assert bool((train_df.loc[train_df["is_synthetic"], "temporal_mask"] == 0).all()), \
    "INVARIANT#1: a synthetic row was not masked out of the temporal axis!"
# masked-out train rows are exactly the rows without a real timestamp (== synthetic here)
assert n_masked_train == int((~train_df["_has_time"]).sum()), \
    "masked count != timestamp-less count"
# test is frozen real -> all masked-in
assert bool((test_df["temporal_mask"] == 1).all()), "frozen test has a masked-out row (unexpected)"
# scaled features of masked-IN rows must lie in [0,1]
for df, nm in [(train_df, "train"), (test_df, "test")]:
    sub = df[df["temporal_mask"] == 1]
    for feat in TEMPORAL_FEATURES:
        v = sub[feat + "_t"].astype(float)
        assert float(v.min()) >= -1e-9 and float(v.max()) <= 1 + 1e-9, f"{nm}.{feat}_t out of [0,1]"

log.info("temporal_mask: train masked-out=%d (synthetic=%d) | test masked-out=%d",
         n_masked_train, n_syn_train, int((test_df['temporal_mask'] == 0).sum()))
print("[OK] Invariant #1 — all %d synthetic rows masked out of the temporal axis" % n_syn_train)

2026-08-07 21:26:40,130 | INFO | temporal_mask: train masked-out=125 (synthetic=125) | test masked-out=0


[OK] Invariant #1 — all 125 synthetic rows masked out of the temporal axis


## 5. Tier-A — persist the canonical temporal feature stream

One row per `record_id`, aligned 1:1 to `train_augmented` / `test_frozen`, carrying the six scaled
features (`*_t`), the `temporal_mask`, and provenance (`platform`, `is_synthetic`, raw `timestamp`).
Objective 3 joins this on `record_id`.

In [6]:
KEEP = (["record_id", "platform", "is_synthetic", "timestamp", "temporal_mask"]
        + [f + "_t" for f in TEMPORAL_FEATURES])

def temporal_frame(df):
    out = df.copy()
    for c in KEEP:
        if c not in out.columns:
            out[c] = None
    return out[KEEP]

temp_train = temporal_frame(train_df)
temp_test  = temporal_frame(test_df)

def write_jsonl(frame, path):
    with open(path, "w", encoding="utf-8") as fh:
        for _, r in frame.iterrows():
            fh.write(json.dumps(r.to_dict(), ensure_ascii=False, default=str) + "\n")

write_jsonl(temp_train, OUTPUT_DIR / "temporal_features_train.jsonl")
write_jsonl(temp_test,  OUTPUT_DIR / "temporal_features_test.jsonl")
for name, frame in [("temporal_features_train", temp_train), ("temporal_features_test", temp_test)]:
    try:
        frame.to_parquet(OUTPUT_DIR / (name + ".parquet"), index=False)
        log.info("wrote %s.parquet", name)
    except Exception as e:
        log.warning("parquet %s skipped (%s); jsonl is source of truth", name, e)

log.info("temporal stream written: train=%d test=%d", len(temp_train), len(temp_test))

2026-08-07 21:26:40,756 | INFO | wrote temporal_features_train.parquet


2026-08-07 21:26:40,772 | INFO | wrote temporal_features_test.parquet


2026-08-07 21:26:40,772 | INFO | temporal stream written: train=1971 test=2768


## 6. Tier-B — assemble the longitudinal drift subset (Reddit/Twitter only, Invariant #2)

Group the frozen Reddit+Twitter rows by user, order chronologically, keep users with **≥ D1 floor (3)**
posts, and chunk each user's history into **`min(10, n_posts)`** contiguous chronological chunks as
the scaffold for Couto's delta-matrix. We record `seq_index`, `chunk_index`, `n_posts_user`,
`n_chunks_user`, and `span_days` (D6 — so genuine longitudinal histories are distinguishable from
same-day harvest bursts). Asserted: the subset has **0 YouTube and 0 synthetic** rows.

In [7]:
drift_src = test_df[test_df["platform"].isin(DRIFT_PLATFORMS) & (~test_df["is_synthetic"])].copy()
assert (~drift_src["platform"].isin(TRAIN_PLATFORMS)).all(), "INVARIANT#2: YouTube row in drift source!"
assert bool((~drift_src["is_synthetic"]).all()), "INVARIANT#2: synthetic row in drift source!"

drift_src["_ts"] = pd.to_datetime(drift_src["timestamp"], utc=True, errors="coerce")
drift_src = drift_src.dropna(subset=["_ts"])

posts_per_user = drift_src.groupby("anonymized_user_id").size()

def threshold_counts(series):
    return {f">={k}": int((series >= k).sum()) for k in (2, 3, 5, FULL_COUTO_MIN_POSTS)}

dist_all     = threshold_counts(posts_per_user)
dist_reddit  = threshold_counts(drift_src[drift_src["platform"] == "reddit"].groupby("anonymized_user_id").size())
dist_twitter = threshold_counts(drift_src[drift_src["platform"] == "twitter"].groupby("anonymized_user_id").size())

viable_users = posts_per_user[posts_per_user >= DRIFT_MIN_POSTS].index
log.info("drift users: total=%d | >=2=%d >=3=%d >=5=%d >=10=%d",
         posts_per_user.size, dist_all[">=2"], dist_all[">=3"], dist_all[">=5"],
         dist_all[">=%d" % FULL_COUTO_MIN_POSTS])
log.info("viable (>= %d posts) users kept = %d", DRIFT_MIN_POSTS, len(viable_users))

2026-08-07 21:26:40,832 | INFO | drift users: total=2006 | >=2=363 >=3=147 >=5=49 >=10=10


2026-08-07 21:26:40,833 | INFO | viable (>= 3 posts) users kept = 147


In [8]:
def chunk_user(n_posts):
    k = min(MAX_CHUNKS, n_posts)
    # contiguous chronological chunks: post i (0-based) -> floor(i*k/n)
    return [int(math.floor(i * k / n_posts)) for i in range(n_posts)], k

drift_rows = []
for uid in viable_users:
    g = drift_src[drift_src["anonymized_user_id"] == uid].sort_values("_ts").reset_index(drop=True)
    n = len(g)
    chunks, k = chunk_user(n)
    span_days = int((g["_ts"].iloc[-1] - g["_ts"].iloc[0]).days)
    for i, (_, r) in enumerate(g.iterrows()):
        drift_rows.append({
            "anonymized_user_id": uid,
            "platform": r["platform"],
            "timestamp": r["timestamp"],
            "post_text_clean": r.get("post_text_clean", r.get("post_text")),
            "tier": int(r["tier"]) if pd.notna(r.get("tier")) else None,
            "record_id": r.get("record_id"),
            "seq_index": i,
            "chunk_index": chunks[i],
            "n_posts_user": n,
            "n_chunks_user": k,
            "span_days": span_days,
        })

drift_df = pd.DataFrame(drift_rows)
# Invariant #2 (belt and suspenders on the emitted artifact)
assert len(drift_df) == 0 or set(drift_df["platform"].unique()) <= DRIFT_PLATFORMS, "drift subset has non-drift platform!"

write_jsonl(drift_df, OUTPUT_DIR / "drift_subset.jsonl")
log.info("drift_subset: %d posts across %d users (>=%d posts each)",
         len(drift_df), drift_df["anonymized_user_id"].nunique() if len(drift_df) else 0, DRIFT_MIN_POSTS)

2026-08-07 21:26:41,835 | INFO | drift_subset: 693 posts across 147 users (>=3 posts each)


## 7. The full-vs-bounded drift verdict

Couto's 10-chunk delta-matrix wants ≥10 posts/user. We count how many users clear that bar; if too few
(< 5), the longitudinal study is reported — honestly — as a **bounded case study with logged N** (the
methodology's own §8.3/§8.5 fallback). The full study unlocks automatically after the deferred data
scale-up, with no code change.

In [9]:
full_viable = int((posts_per_user >= FULL_COUTO_MIN_POSTS).sum())
verdict = "full" if full_viable >= FULL_COUTO_MIN_USERS else "bounded_case_study"
span_summary = {}
if len(drift_df):
    spans = drift_df.groupby("anonymized_user_id")["span_days"].first()
    span_summary = {
        "median_span_days": float(spans.median()),
        "max_span_days": int(spans.max()),
        "users_span_0_days_same_session": int((spans == 0).sum()),
        "users_span_ge_7_days": int((spans >= 7).sum()),
    }

log.info("DRIFT VERDICT = %s (users with >=%d posts = %d; need >=%d for 'full')",
         verdict, FULL_COUTO_MIN_POSTS, full_viable, FULL_COUTO_MIN_USERS)
print("DRIFT VERDICT:", verdict.upper(), "| full-viable users:", full_viable)
print("span summary:", span_summary)

2026-08-07 21:26:41,851 | INFO | DRIFT VERDICT = full (users with >=10 posts = 10; need >=5 for 'full')


DRIFT VERDICT: FULL | full-viable users: 10
span summary: {'median_span_days': 0.0, 'max_span_days': 562, 'users_span_0_days_same_session': 106, 'users_span_ge_7_days': 38}


## 8. Temporal/drift report + posting-time histogram

`temporal_report.json` records the mask counts, scaler ranges, the posting-time (hour) histogram (the
Saeed & Cha "late-night distress window" data, computed on real rows only), the posts-per-user
distribution per platform, the chosen floor, and the **full-vs-bounded verdict with logged N**.

In [10]:
real_train_hours = pd.to_numeric(train_df.loc[train_df["temporal_mask"] == 1, "raw_hour"], errors="coerce").dropna().astype(int)
real_test_hours  = pd.to_numeric(test_df.loc[test_df["temporal_mask"] == 1, "raw_hour"], errors="coerce").dropna().astype(int)
hour_hist_all = Counter(pd.concat([real_train_hours, real_test_hours]).tolist())
hour_hist = {h: int(hour_hist_all.get(h, 0)) for h in range(24)}

drift_tier_dist = {}
if len(drift_df):
    drift_tier_dist = {int(k): int(v) for k, v in drift_df["tier"].value_counts().sort_index().items()}

report = {
    "part": "stage1_part1.4_temporal_drift",
    "version": PART_VERSION,
    "principle": "P1 (no human); P2 (temporal auxiliary); P3 (train=youtube, test=reddit+twitter); "
                 "P5 (drift only where data supports it);  tiered temporal",
    "decisions": {
        "D1_drift_min_posts": DRIFT_MIN_POSTS, "D2_max_chunks": MAX_CHUNKS,
        "D3_minmax_refit": "TRAIN real rows only", "D4_platforms": sorted(DRIFT_PLATFORMS),
        "D5_ruminative_repetition": "deferred to Objective 2 preprocessing",
        "D6_keep_same_day_bursts": True,
    },
    "temporal_features": TEMPORAL_FEATURES,
    "scaler_ranges_train_fit": {k: [v[0], v[1]] for k, v in SCALER.items()},
    "n_train_rows": int(len(train_df)), "n_test_rows": int(len(test_df)),
    "train_real": int((~train_df["is_synthetic"]).sum()), "train_synthetic": n_syn_train,
    "temporal_mask": {
        "train_masked_in": int((train_df["temporal_mask"] == 1).sum()),
        "train_masked_out": n_masked_train,
        "test_masked_in": int((test_df["temporal_mask"] == 1).sum()),
        "test_masked_out": int((test_df["temporal_mask"] == 0).sum()),
        "synthetic_all_masked_out": bool((train_df.loc[train_df["is_synthetic"], "temporal_mask"] == 0).all()),
    },
    "posting_hour_histogram_real": hour_hist,
    "drift": {
        "source": "frozen reddit+twitter test, real rows only",
        "leakage_check_zero_youtube_zero_synthetic": True,
        "unique_users_total": int(posts_per_user.size),
        "users_by_threshold_all": dist_all,
        "users_by_threshold_reddit": dist_reddit,
        "users_by_threshold_twitter": dist_twitter,
        "viability_floor_posts": DRIFT_MIN_POSTS,
        "viable_users_kept": int(len(viable_users)),
        "drift_subset_posts": int(len(drift_df)),
        "max_posts_per_user": int(posts_per_user.max()) if posts_per_user.size else 0,
        "full_couto_min_posts": FULL_COUTO_MIN_POSTS,
        "full_couto_viable_users": full_viable,
        "verdict": verdict,
        "span_summary": span_summary,
        "drift_subset_tier_dist": drift_tier_dist,
    },
}
(OUTPUT_DIR / "temporal_report.json").write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(report, indent=2, ensure_ascii=False))

{
  "part": "stage1_part1.4_temporal_drift",
  "version": "1.0",
  "principle": "P1 (no human); P2 (temporal auxiliary); P3 (train=youtube, test=reddit+twitter); P5 (drift only where data supports it);  tiered temporal",
  "decisions": {
    "D1_drift_min_posts": 3,
    "D2_max_chunks": 10,
    "D3_minmax_refit": "TRAIN real rows only",
    "D4_platforms": [
      "reddit",
      "twitter"
    ],
    "D5_ruminative_repetition": "deferred to Objective 2 preprocessing",
    "D6_keep_same_day_bursts": true
  },
  "temporal_features": [
    "month",
    "day",
    "hour",
    "weekday",
    "is_working_hour",
    "is_weekend"
  ],
  "scaler_ranges_train_fit": {
    "month": [
      1.0,
      12.0
    ],
    "day": [
      1.0,
      31.0
    ],
    "hour": [
      0.0,
      23.0
    ],
    "weekday": [
      0.0,
      6.0
    ],
    "is_working_hour": [
      0.0,
      1.0
    ],
    "is_weekend": [
      0.0,
      1.0
    ]
  },
  "n_train_rows": 1971,
  "n_test_rows": 2768,
  "train

## 9. Temporal/drift card & This step is complete when

**notebook 04 is complete when:** the temporal feature stream is written for train+test (six `*_t` features
⊆ [0,1] + `temporal_mask`); every synthetic row is masked out (count == 35); the drift subset is built
from Reddit+Twitter only (0 YouTube, 0 synthetic) with chronological order + adaptive chunks + span; and
the report records the posts-per-user distribution and the explicit full-vs-bounded verdict with logged N.
**No human anywhere.**

In [11]:
print("=" * 64); print("Objective 1 · notebook 04 — TEMPORAL / DRIFT CARD"); print("=" * 64)
print("temporal stream train/test :", len(temp_train), "/", len(temp_test))
print("synthetic rows masked out  :", n_syn_train, "(all masked:",
      bool((train_df.loc[train_df['is_synthetic'], 'temporal_mask'] == 0).all()), ")")
print("drift subset posts / users :", len(drift_df), "/",
      (drift_df['anonymized_user_id'].nunique() if len(drift_df) else 0))
print("drift users >=2/>=3/>=5/>=10:", dist_all[">=2"], dist_all[">=3"], dist_all[">=5"],
      dist_all[">=%d" % FULL_COUTO_MIN_POSTS])
print("DRIFT VERDICT              :", verdict.upper(), "(full-viable users:", full_viable, ")")
dod = all([
    (OUTPUT_DIR / "temporal_features_train.jsonl").exists(),
    (OUTPUT_DIR / "temporal_features_test.jsonl").exists(),
    (OUTPUT_DIR / "drift_subset.jsonl").exists(),
    (OUTPUT_DIR / "temporal_report.json").exists(),
    bool((train_df.loc[train_df["is_synthetic"], "temporal_mask"] == 0).all()),
    n_masked_train == n_syn_train,
    (len(drift_df) == 0 or set(drift_df["platform"].unique()) <= DRIFT_PLATFORMS),
])
print("\nCompletion check:", "PASS" if dod else "INCOMPLETE")

Objective 1 · notebook 04 — TEMPORAL / DRIFT CARD
temporal stream train/test : 1971 / 2768
synthetic rows masked out  : 125 (all masked: True )
drift subset posts / users : 693 / 147
drift users >=2/>=3/>=5/>=10: 363 147 49 10
DRIFT VERDICT              : FULL (full-viable users: 10 )

Completion check: PASS
